# Qatar TB — TB vs Normal Classification with DenseNet121

*A writeup of how I built and investigated a TB classifier on the Qatar TB chest X-ray dataset — including the part where I suspected the model was cheating, and what actually turned out to be going on.*

**What this notebook is:** I trained a DenseNet121 on the Qatar TB dataset (700 TB / 3500 Normal chest X-rays), then investigated a suspicious result — the model had perfect precision but partial recall. I measured the raw image files, found a class-correlated color artifact, and finally used Grad-CAM to see where the model actually looks.

**Context:** in my research project this dataset is an *external test set* — the actual training happens on NIH ChestX-ray14. This notebook's job was to characterize the dataset honestly: its artifacts, its signal strength, and what a model trained on it really does.

In [ ]:
import os
print("Working dir:", os.getcwd())
print("Files here:", os.listdir("."))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import time
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from PIL import Image
from glob import glob
from collections import Counter
import numpy as np

**Setup.** I fixed the random seed (42) so every run is reproducible and directly comparable — early on I noticed the same code could give very different results depending on random initialization and shuffling.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

**Data.** Qatar TB (Kaggle `tawsifurrahman/tuberculosis-tb-chest-xray-dataset`): 700 TB / 3500 Normal chest X-rays (5:1 imbalance), all 512×512 PNGs, from Hamad Medical Corporation (Qatar). I split 70/15/15 → train/dev/test = 2940/630/630.

In [ ]:
data_path = "/kaggle/input/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset/TB_Chest_Radiography_Database"

tb_paths = glob(f"{data_path}/Tuberculosis/*.png")
normal_paths = glob(f"{data_path}/Normal/*.png")
print(len(tb_paths), len(normal_paths))   # expect: 700 3500

In [ ]:
data = []
for path in tb_paths:
    data.append((path, 1))     # TB = 1
for path in normal_paths:
    data.append((path, 0))     # Normal = 0

print(len(data))               # expect: 4200
print(data[0], data[-1])

In [ ]:
random.seed(42)
random.shuffle(data)

n = len(data)
train = data[:int(n*0.7)]
dev   = data[int(n*0.7):int(n*0.85)]
test  = data[int(n*0.85):]
print(f"train: {len(train)}, dev: {len(dev)}, test: {len(test)}")  # 2940 630 630

**The grayscale decision.** I convert every image to grayscale inside the Dataset. Why this matters: the raw files carry a class-correlated color artifact (I measure it below) — about half of the TB images are colored, while every Normal image is flat gray. By forcing grayscale, the model can never see color, so any performance it reaches isn't color-based cheating. I verify R==G==B in the actual tensor later.

In [ ]:
class ChestXRayDataset(Dataset):
    def __init__(self, data):
        self.data = data
        self.transforms = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        path, label = self.data[index]
        img = Image.open(path).convert("L")
        img_tensor = self.transforms(img)
        label_tensor = torch.tensor(label)
        return img_tensor, label_tensor

In [ ]:
train_ds = ChestXRayDataset(train)
dev_ds   = ChestXRayDataset(dev)
test_ds  = ChestXRayDataset(test)

train_dloader = DataLoader(train_ds, batch_size=32, shuffle=True)
dev_dloader   = DataLoader(dev_ds,   batch_size=32, shuffle=False)
test_dloader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

**Sanity checks before training.** Class balance: 2445 Normal / 495 TB (83.2% / 16.8%) — matches the source 5:1 ratio. Batch shape [32,3,224,224] with identical channels — color is dead in the model's input.

In [ ]:
# Sanity 1: class balance
counts = Counter()
for img, lbl in train_dloader:
    counts.update(lbl.tolist())
print(counts)   # expect: Counter({0: ~2445, 1: ~495})

In [ ]:
# Sanity 2: one batch shape + color check
img, lbl = next(iter(train_dloader))
print("Batch shape:", img.shape)                  # [32, 3, 224, 224]
print("Channels identical:", torch.allclose(img[0,0], img[0,1]))  # True

# 

**Model.** DenseNet121 pretrained on ImageNet (transfer learning), with the classifier head swapped to `Linear(1024, 2)`. I started from pretrained weights since the dataset is small.

In [ ]:
device= "cuda" if torch.cuda.is_available() else "cpu"    #this line will choose the gpu to train the model if its available and its not it will train on the cpu  

model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
model.classifier = nn.Linear(in_features=1024, out_features=2, bias=True)
model = model.to(device)
print(model.classifier)   # expect: Linear(1024, 2)

In [ ]:
from collections import Counter
counts = Counter()
for img, lbl in train_dloader:
    counts.update(lbl.tolist())
print(counts)

**Training.** lr=1e-4, gradient clipping (max_norm=1.0), 5 epochs, seeded. One honest note: I compute a class-imbalance `weights` tensor here but don't pass it to `CrossEntropyLoss()` — so this run is effectively *unweighted*. I left it that way intentionally; `nn.CrossEntropyLoss(weight=weights)` is the class-weighted version.

In [ ]:
num_epochs = 5
optimizer = optim.Adam(model.parameters(), lr=1e-4)
weights = torch.tensor([1.0, 2445/495]).to(device)
loss_fn = nn.CrossEntropyLoss()

total_start = time.time()

for epoch in range(num_epochs):
    epoch_start = time.time()
    running_loss = 0.0
    n_batches = 0

    model.train()

    for img_tensor, label_tensor in train_dloader:
        img_tensor = img_tensor.to(device)
        label_tensor = label_tensor.to(device)

        optimizer.zero_grad()
        outputs = model(img_tensor)
        loss = loss_fn(outputs, label_tensor)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        n_batches += 1

    avg_loss = running_loss / n_batches
    epoch_time = time.time() - epoch_start
    print(f"Epoch {epoch+1}/{num_epochs} | avg loss: {avg_loss:.4f} | time: {epoch_time:.1f}s")

total_time = time.time() - total_start
print(f"Total training time: {total_time:.1f}s ({total_time/60:.1f} min)")

**Train accuracy — reference only.** I measured it on the training set, which the model has already seen, so it doesn't tell me anything about generalization. The held-out evaluation further down is the number that matters.

In [ ]:
correct = total = 0
model.eval()
with torch.no_grad():
    for img, lbl in train_dloader:
        img, lbl = img.to(device), lbl.to(device)
        preds = model(img).argmax(1)
        correct += (preds == lbl).sum().item()
        total += lbl.size(0)
print(f"Train accuracy: {correct/total:.3f}")

**Visual check.** I plotted one TB and one Normal X-ray side by side. Beyond the disease itself, the two images look different — which is what first made me suspect something was off in the files, not just in the model.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(Image.open(tb_paths[1]))       # a TB image
axes[0].set_title("TB")
axes[1].imshow(Image.open(normal_paths[1]))   # a Normal image
axes[1].set_title("Normal")
plt.show()

**Why I investigated color.** The held-out result below showed *perfect precision, partial recall* (TB precision 1.00 / recall 0.60) — zero false positives. Real classifiers usually make mistakes in both directions, so a wall of zero false positives looked suspicious to me: maybe the model found an easy file-level trick instead of learning TB. I decided to look at the raw files themselves.

In [ ]:
import numpy as np
from PIL import Image

def color_score(paths):
    scores = []
    for p in paths[:100]:
        img = np.array(Image.open(p).convert("RGB")).astype(float)
        # how different are R, G, B channels? gray = 0, colored = big
        score = np.abs(img[...,0] - img[...,1]).mean() + np.abs(img[...,1] - img[...,2]).mean()
        scores.append(score)
    return np.mean(scores)

print("TB color score:", color_score(tb_paths))
print("Normal color score:", color_score(normal_paths))

**Inspecting one TB file.** I checked the image mode and channel statistics to see whether the image was truly grayscale or carried color.

In [ ]:
from PIL import Image
import numpy as np

p = tb_paths[9]
img = Image.open(p)
print("Mode:", img.mode, "| Size:", img.size)

arr = np.array(img)
print("Shape:", arr.shape, "dtype:", arr.dtype)

if arr.ndim == 3:
    for c, name in enumerate(["R", "G", "B"]):
        print(f"{name}: mean {arr[..., c].mean():.1f} | min {arr[..., c].min()} | max {arr[..., c].max()}")

    diff = np.abs(arr[..., 0].astype(float) - arr[..., 1].astype(float))
    print(f"Mean |R-G| over whole image: {diff.mean():.2f}")
    print(f"Max |R-G| anywhere: {diff.max():.0f}")

    h, w = arr.shape[:2]
    center = arr[h//4:3*h//4, w//4:3*w//4]
    center_diff = np.abs(center[..., 0].astype(float) - center[..., 1].astype(float)).mean()
    print(f"Color in CENTER (lung area): {center_diff:.2f}")
    print(f"Color in full image:         {diff.mean():.2f}")

**The decisive file-level measurement.** I scanned 200 images per class: **TB = mix of `L` and `RGB` modes, about half genuinely colored** (102/200 with color score > 5, mean 9.82); **Normal = ALL flat gray** (0/200, score 0.00). So the color artifact is real and class-correlated — a model trained on raw RGB could hit near-perfect accuracy by color alone. That's a genuine dataset flaw worth documenting (it became a Limitations paragraph in my paper). The open question was: does *my* model (grayscale-forced) actually use it?

In [ ]:
from PIL import Image
import numpy as np
from collections import Counter

def scan(paths, name, n=200):
    modes = Counter()
    scores = []
    for p in paths[:n]:
        img = Image.open(p)
        modes[img.mode] += 1
        arr = np.array(img.convert("RGB")).astype(float)
        scores.append((np.abs(arr[...,0]-arr[...,1]).mean() + np.abs(arr[...,1]-arr[...,2]).mean()) / 2)
    print(f"{name}: modes={dict(modes)}")
    print(f"{name}: color score mean={np.mean(scores):.2f} | min={min(scores):.2f} | max={max(scores):.2f}")
    print(f"{name}: #images with score>5 = {sum(1 for s in scores if s>5)} / {n}")
    print()

scan(tb_paths, "TB")
scan(normal_paths, "Normal")

**Next suspect: brightness.** I checked it next and ruled it out — TB 127.2 vs Normal 131.3 (~3% difference, normal X-ray variation). Color was the artifact; brightness wasn't.

In [ ]:
import numpy as np
from PIL import Image

tb_b = np.mean([np.array(Image.open(p).convert("L")).mean() for p in tb_paths[:200]])
nm_b = np.mean([np.array(Image.open(p).convert("L")).mean() for p in normal_paths[:200]])
print(f"TB mean brightness: {tb_b:.1f}")
print(f"Normal mean brightness: {nm_b:.1f}")

**Does the model actually see the artifact?** I ran one image through the Dataset transform: channels identical, R==G==B exactly. The grayscale transform kills color before the model ever sees it — the artifact exists in the files, but this model can't use it.

In [ ]:
# Fresh dataset from your CURRENT class
ds_test = ChestXRayDataset(train)
img, lbl = ds_test[0]   # one image through YOUR transform

print("Tensor shape:", img.shape)
print("Channel 0 == Channel 1:", torch.allclose(img[0], img[1]))
print("Channel 1 == Channel 2:", torch.allclose(img[1], img[2]))

# per-channel means (if identical → color dead)
print("R mean:", img[0].mean().item())
print("G mean:", img[1].mean().item())
print("B mean:", img[2].mean().item())

**BatchNorm: train() vs eval() mode.** DenseNet121 is full of BatchNorm layers — `train()` mode uses live batch statistics, `eval()` mode uses running averages accumulated during training. I tested whether predictions actually differ between the two modes — a real but usually small effect worth keeping in mind whenever metrics are compared across `train()` and `eval()`.

In [ ]:
model.eval()
img, lbl = next(iter(train_dloader))
img, lbl = img.to(device), lbl.to(device)

with torch.no_grad():
    preds_eval = model(img).argmax(1)

model.train()
with torch.no_grad():
    preds_train_mode = model(img).argmax(1)

print("Predictions match between train()/eval() modes:", torch.equal(preds_eval, preds_train_mode))
print("Eval mode accuracy on this batch:", (preds_eval == lbl).float().mean().item())
print("Train mode accuracy on this batch:", (preds_train_mode == lbl).float().mean().item())

**The real result — held-out test set.** ~93% accuracy, TB precision 1.00 / recall 0.60. This pattern (perfect precision, partial recall) is consistent with two explanations: (a) the model exploits the file artifact; (b) any model trained on 5:1 imbalanced data without class weights becomes cautious — it only calls TB when very confident → high precision, low recall. Numbers alone can't tell these apart, which is why I went to Grad-CAM.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for img, lbl in test_dloader:
        img, lbl = img.to(device), lbl.to(device)
        preds = model(img).argmax(1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(lbl.cpu().tolist())

print(classification_report(all_labels, all_preds, target_names=["Normal", "TB"]))
print(confusion_matrix(all_labels, all_preds))

**Collecting cases for Grad-CAM.** I grabbed true positives (correctly caught TB) and false negatives (missed TB) from the held-out test — comparing *where* the model looks in each shows whether it uses real pathology or an artifact.

In [ ]:
tp_examples, fn_examples = [], []
model.eval()

for img_batch, lbl_batch in test_dloader:
    img_batch, lbl_batch = img_batch.to(device), lbl_batch.to(device)
    with torch.no_grad():
        preds = model(img_batch).argmax(1)

    for i in range(img_batch.size(0)):
        true_lbl = lbl_batch[i].item()
        pred_lbl = preds[i].item()
        if true_lbl == 1 and pred_lbl == 1 and len(tp_examples) < 5:
            tp_examples.append(img_batch[i:i+1].clone())
        elif true_lbl == 1 and pred_lbl == 0 and len(fn_examples) < 5:
            fn_examples.append(img_batch[i:i+1].clone())

    if true_lbl == 1 and pred_lbl == 1 and len(tp_examples) < 5:
        tp_examples.append(img_batch[i:i+1].clone())
    elif true_lbl == 1 and pred_lbl == 0 and len(fn_examples) < 8:   # collect more FNs, in case there are few
        fn_examples.append(img_batch[i:i+1].clone())
    
    if len(tp_examples) >= 5 and len(fn_examples) >= 3:   # don't require all 5 FNs to stop
        break

print(f"Collected {len(tp_examples)} true positives, {len(fn_examples)} false negatives")

**Grad-CAM.** It uses the gradients flowing back into the last convolutional layer to build a heatmap of the regions that most influenced the prediction. Two implementation notes from my debugging: I clear leftover hooks before registering (re-running hook registration without clearing accumulates hooks and causes crashes), and I register the hook on the output tensor inside a forward hook to avoid DenseNet121's inplace-relu autograd conflict.

In [ ]:
import torch.nn.functional as F
import matplotlib.pyplot as plt

target_layer = model.features

# clear ALL leftover hooks from every previous attempt
target_layer._forward_hooks.clear()
target_layer._forward_pre_hooks.clear()
if hasattr(target_layer, "_backward_hooks"):
    target_layer._backward_hooks.clear()
if hasattr(target_layer, "_backward_pre_hooks"):
    target_layer._backward_pre_hooks.clear()

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._forward_hook)

    def _forward_hook(self, module, input, output):
        self.activations = output
        output.register_hook(self._save_gradient)

    def _save_gradient(self, grad):
        self.gradients = grad.detach()

    def generate(self, input_tensor, class_idx):
        self.model.eval()
        output = self.model(input_tensor)
        self.model.zero_grad()
        score = output[0, class_idx]
        score.backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations.detach()).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(224, 224), mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, output.softmax(dim=1)[0, class_idx].item()

gradcam = GradCAM(model, target_layer)

**What I found.** If the model were cheating, the heatmaps would concentrate on artifact regions (boxes, borders) even for correct calls. Instead: true positives heat over **lung fields**, avoiding the white de-identification boxes; false negatives are **diffuse/uncertain** — no confident evidence, not confidently-wrong evidence. That is the signature of a conservative model looking at real anatomy, not a shortcut-exploiting cheater. I was wrong about the cheating hypothesis.

In [ ]:
def plot_gradcam_row(examples, title_prefix):
    n = len(examples)
    fig, axes = plt.subplots(2, n, figsize=(4*n, 8))
    for i, img_tensor in enumerate(examples):
        img_tensor = img_tensor.clone().requires_grad_(False)
        cam, conf = gradcam.generate(img_tensor, class_idx=1)
        orig = img_tensor[0, 0].detach().cpu().numpy()

        axes[0, i].imshow(orig, cmap="gray")
        axes[0, i].set_title(f"{title_prefix} #{i+1}")
        axes[0, i].axis("off")

        axes[1, i].imshow(orig, cmap="gray")
        axes[1, i].imshow(cam, cmap="jet", alpha=0.45)
        axes[1, i].set_title(f"TB conf: {conf:.2f}")
        axes[1, i].axis("off")

    plt.suptitle(f"{title_prefix} — Grad-CAM (top: original, bottom: overlay)")
    plt.tight_layout()
    plt.show()

plot_gradcam_row(tp_examples, "True Positive")
if len(fn_examples) > 0:
    plot_gradcam_row(fn_examples, "False Negative")
else:
    print("No false negatives collected this run.")

## Verdict

1. **Color artifact: real** (measured, reproducible) — but *latent*: the model can't see it (grayscale verified R==G==B).
2. **The "too-clean" numbers = class-imbalance behavior, not cheating.** Any unweighted model on 5:1 data behaves this way.
3. **The grayscale signal is weak.** After grayscale, pixel-statistic features separate the classes only modestly (best single feature = entropy, AUC 0.84) — there is no strong grayscale-surviving shortcut.
4. **Grad-CAM: the model looks at real anatomy.** It's conservative, not a cheater.
5. **Qatar stays external-test-with-caveat — never training.** Report the stable seeded numbers with the caveat.

## What I learned

- **Evaluate on held-out data.** Training-set numbers describe memorization, not generalization.
- **Check the mechanism before trusting a number.** The "too-clean" matrix looked like cheating; Grad-CAM showed it was caution. Numbers alone can't separate the two.